# 013a: Multi-Model LLM Base Rate Researcher (Primary Prototype)

Builds and tests the multi-model base rate researcher (Option 3).
Independently queries GPT-5.2 and Claude 4.6 for base rate analysis,
then synthesizes with a summarizer model (o4-mini or Claude Sonnet 4.6).

**This is the primary notebook — it produces the production functions for `research.py`.**

**Production target**: `research.py` (new file)

**Functions to extract**:

| Function | Target | Cell |
|----------|--------|------|
| `_build_base_rate_prompt()` | `research.py` | 3 |
| `_query_model()` | `research.py` | 4 |
| `_synthesize_analyses()` | `research.py` | 5 |
| `get_base_rate_research()` | `research.py` | 6 |

**Success criteria**: Multi-model synthesis produces base rate estimates that are (a) more reliable
than either model alone, and (b) usefully flag uncertainty when models disagree.

**Cost estimate**: ~$0.25-0.75 total for full test run

## Update Procedure

| Component | Source File | Notes |
|---|---|---|
| LLM calls | `openai.AsyncOpenAI` via OpenRouter | Production should use `GeneralLlm` |
| Model strings | OpenRouter paths | e.g. `openai/gpt-5.2` (no `openrouter/` prefix) |
| Web search | `:online` suffix | Appended for web-grounded queries |
| Production target | `research.py` | New standalone module |


In [1]:
# EXPLORATION ONLY
# Cell 0: Inputs — Edit these before running

# --- Test Question IDs ---
TEST_QUESTION_IDS = [
    42562,  # Binary: Ducks playoffs (clear reference class)
    # Add more question IDs here:
    # Binary with ambiguous reference class (geopolitical event)
    # Numeric with historical trend data
    # Multiple choice
    # Edge case: novel event with no historical precedent
]

# --- External News/Research File ---
NEWS_FILE = "../data/Run Log News Summaries/42562_News_Summary_03-15-2026.txt"

# --- Model Configuration (OpenRouter paths without 'openrouter/' prefix) ---
MODEL_A = "openai/gpt-5.2"              # Frontier model A
MODEL_B = "anthropic/claude-opus-4-6"    # Frontier model B
SYNTHESIZER = "openai/o4-mini"            # Synthesis model
SYNTHESIZER_ALT = "anthropic/claude-sonnet-4-6"  # A/B test alternate

# --- LLM Parameters ---
TEMPERATURE = 0.3
TIMEOUT = 60       # seconds per LLM call
USE_WEB_SEARCH = False  # Set True to append :online suffix

print("=== Inputs ===")
print(f"Questions: {TEST_QUESTION_IDS}")
print(f"News file: {NEWS_FILE}")
print(f"Model A:   {MODEL_A}")
print(f"Model B:   {MODEL_B}")
print(f"Synth:     {SYNTHESIZER}")
print(f"Synth alt: {SYNTHESIZER_ALT}")
print(f"Temp: {TEMPERATURE}, Timeout: {TIMEOUT}s, Web search: {USE_WEB_SEARCH}")


=== Inputs ===
Questions: [42562]
News file: ../data/Run Log News Summaries/42562_News_Summary_03-15-2026.txt
Model A:   openai/gpt-5.2
Model B:   anthropic/claude-opus-4-6
Synth:     openai/o4-mini
Synth alt: anthropic/claude-sonnet-4-6
Temp: 0.3, Timeout: 60s, Web search: False


In [2]:
# EXPLORATION ONLY
# Cell 1: Setup & Imports

import os
import json
import asyncio
import time
from types import SimpleNamespace

import requests
import nest_asyncio
from openai import AsyncOpenAI

nest_asyncio.apply()

client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

print("OpenRouter client initialized.")
print(f"OPENROUTER_API_KEY set: {'OPENROUTER_API_KEY' in os.environ}")


OpenRouter client initialized.
OPENROUTER_API_KEY set: True


In [3]:
# EXPLORATION ONLY
# Cell 2: Fetch Test Questions from Metaculus API

def fetch_metaculus_question(question_id):
    """Fetch question data from Metaculus API and return as SimpleNamespace."""
    url = f"https://www.metaculus.com/api/questions/{question_id}/"
    headers = {"Authorization": f"Token {os.environ.get('METACULUS_BOT_API_TOKEN', '')}"}
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    data = resp.json()
    return SimpleNamespace(
        id_of_question=data.get("id"),
        question_text=data.get("title", ""),
        resolution_criteria=data.get("resolution_criteria", ""),
        fine_print=data.get("fine_print", ""),
        background_info=data.get("description", ""),
        question_type=data.get("type", "binary"),
        community_prediction=data.get("community_prediction", {}).get("full", {}).get("q2"),
    )

test_questions = []
for qid in TEST_QUESTION_IDS:
    try:
        q = fetch_metaculus_question(qid)
        test_questions.append(q)
        print(f"Q{q.id_of_question} ({q.question_type}): {q.question_text[:80]}...")
    except Exception as e:
        print(f"Failed to load Q{qid}: {e}")

print(f"\nTotal test questions: {len(test_questions)}")


Q42562 (numeric): What will the percent change in US employment share of janitors and cleaners be ...

Total test questions: 1


In [4]:
# EXPLORATION ONLY
# Cell 2b: Load News/Research from External File

with open(NEWS_FILE, encoding='utf-8') as f:
    news_summary = f.read()

research = news_summary

print(f"Loaded: {NEWS_FILE}")
print(f"Length: {len(research)} characters")
print(f"Preview: {research[:200]}...")


Loaded: ../data/Run Log News Summaries/42562_News_Summary_03-15-2026.txt
Length: 14632 characters
Preview: 2026-03-15 15:36:57,925 - main - INFO - Found Research for URL https://www.metaculus.com/questions/42562:
Here are the relevant news articles:

**Ducks' Playoff Hopes Surge as 2026 Season Nears Conclu...


In [5]:
# PRODUCTION TARGET: research.py::_build_base_rate_prompt
# Cell 3: Build Base Rate Prompt

def _build_base_rate_prompt(
    question_text: str,
    resolution_criteria: str = "",
    fine_print: str = "",
    research_context: str = "",
) -> str:
    """Build the structured prompt for base rate analysis.
    
    Args:
        research_context: News/research text from AskNews or external file.
            When provided, the model uses this as evidence alongside its
            training data for base rate estimation.
    """
    context_parts = [f"Question: {question_text}"]
    if resolution_criteria:
        context_parts.append(f"Resolution criteria: {resolution_criteria}")
    if fine_print:
        context_parts.append(f"Fine print: {fine_print}")
    if research_context:
        context_parts.append(
            f"\nRelevant news and research:\n{research_context}"
        )
    context = "\n".join(context_parts)

    return (
        "You are a base rate analyst for a professional forecasting team.\n"
        "Given the following question and any available research context, "
        "provide a detailed base rate analysis:\n\n"
        "1. Identify the most relevant reference class(es)\n"
        "2. Estimate historical frequency: how often has this type of event occurred?\n"
        "3. Provide the numerator (events of interest) and denominator (opportunities)\n"
        "4. Note any trend (increasing, decreasing, stable)\n"
        "5. State key caveats or differences between the reference class and "
        "this specific question\n\n"
        f"{context}"
    )

# Test: verify prompt looks reasonable (with research context)
if test_questions:
    sample_prompt = _build_base_rate_prompt(
        test_questions[0].question_text,
        test_questions[0].resolution_criteria,
        test_questions[0].fine_print,
        research_context=research[:500],  # preview with truncated research
    )
    print(f"Prompt length (with research): {len(sample_prompt)} chars")
    print(f"---\n{sample_prompt}\n---")
    
    # Also show without research for comparison
    sample_no_research = _build_base_rate_prompt(
        test_questions[0].question_text,
        test_questions[0].resolution_criteria,
        test_questions[0].fine_print,
    )
    print(f"\nPrompt length (no research): {len(sample_no_research)} chars")


Prompt length (with research): 1185 chars
---
You are a base rate analyst for a professional forecasting team.
Given the following question and any available research context, provide a detailed base rate analysis:

1. Identify the most relevant reference class(es)
2. Estimate historical frequency: how often has this type of event occurred?
3. Provide the numerator (events of interest) and denominator (opportunities)
4. Note any trend (increasing, decreasing, stable)
5. State key caveats or differences between the reference class and this specific question

Question: What will the percent change in US employment share of janitors and cleaners be in the following years relative to 2025? (2030)

Relevant news and research:
2026-03-15 15:36:57,925 - main - INFO - Found Research for URL https://www.metaculus.com/questions/42562:
Here are the relevant news articles:

**Ducks' Playoff Hopes Surge as 2026 Season Nears Conclusion**
As of March 15, 2026, the Anaheim Ducks have accumulated 75 po

In [6]:
# PRODUCTION TARGET: research.py::_query_model
# Cell 4: Query a Single Model
# NOTE: Notebook uses raw AsyncOpenAI. Production transfer to research.py
#       should replace with GeneralLlm for consistency with bot framework.

async def _query_model(
    prompt: str,
    model: str,
    temperature: float = 0.3,
    timeout: int = 60,
) -> str:
    """Query a single model via OpenRouter and return its response."""
    response = await asyncio.wait_for(
        client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
        ),
        timeout=timeout,
    )
    return response.choices[0].message.content

# Test: run on one question with Model A
if test_questions:
    test_prompt = _build_base_rate_prompt(
        test_questions[0].question_text,
        test_questions[0].resolution_criteria,
    )
    print(f"Querying {MODEL_A}...")
    start = time.time()
    model_a_response = await _query_model(test_prompt, MODEL_A, TEMPERATURE, TIMEOUT)
    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s ({len(model_a_response)} chars)")
    print(f"---\n{model_a_response[:500]}\n---")


Querying openai/gpt-5.2...
Done in 53.0s (7244 chars)
---
## 1) Most relevant reference class(es)

**Primary reference class (directly on-point):**
- **US employment share of “Janitors and Cleaners, Except Maids and Housekeeping Cleaners”** over **5-year horizons**, using BLS occupational employment (OEWS/OES) for the occupation and a consistent measure of **total US employment** (often OEWS all-occupation employment, or CPS/BLS total employment).
- The question is explicitly about **share** (occupation employment / total employment), so the reference 
---


In [7]:
# PRODUCTION TARGET: research.py::_synthesize_analyses
# Cell 5: Synthesize Two Independent Analyses

async def _synthesize_analyses(
    analysis_a: str,
    analysis_b: str,
    question_text: str,
    model: str,
    temperature: float = 0.2,
    timeout: int = 40,
) -> str:
    """Synthesize two independent analyses into a consensus report."""
    prompt = (
        "You are synthesizing two independent base rate analyses "
        "for a forecasting question.\n"
        "Produce a concise summary that:\n"
        "- States the consensus base rate estimate (if both agree)\n"
        "- Flags disagreements with both perspectives\n"
        "- Provides a final best-estimate rate with confidence "
        "(high/medium/low)\n"
        "- Notes key caveats\n\n"
        f"Question: {question_text}\n\n"
        f"## Analysis A:\n{analysis_a}\n\n"
        f"## Analysis B:\n{analysis_b}"
    )
    return await _query_model(prompt, model, temperature, timeout)

# Test: get Model B response, then synthesize
if test_questions:
    test_prompt = _build_base_rate_prompt(
        test_questions[0].question_text,
        test_questions[0].resolution_criteria,
    )
    print(f"Querying {MODEL_B}...")
    start = time.time()
    model_b_response = await _query_model(test_prompt, MODEL_B, TEMPERATURE, TIMEOUT)
    elapsed = time.time() - start
    print(f"Model B done in {elapsed:.1f}s ({len(model_b_response)} chars)")

    print(f"\nSynthesizing with {SYNTHESIZER}...")
    start = time.time()
    synthesis = await _synthesize_analyses(
        model_a_response, model_b_response,
        test_questions[0].question_text,
        SYNTHESIZER,
    )
    elapsed = time.time() - start
    print(f"Synthesis done in {elapsed:.1f}s")
    print(f"---\n{synthesis}\n---")


Querying anthropic/claude-opus-4-6...
Model B done in 30.4s (4432 chars)

Synthesizing with openai/o4-mini...
Synthesis done in 12.3s
---
Consensus Base-Rate Estimate  
Both analyses agree that the employment share of janitors and cleaners is likely to drift slightly downward over 2025–2030, with typical 5-year relative changes in the single-digit negative range.

Points of Disagreement  
• Analysis A offers a broad historical rule-of-thumb of –10% to +5% (center mildly negative) and emphasizes cyclical bumps, suggesting a small decline unless a recession or rebound intervenes.  
• Analysis B narrows that to –2% to –5% (central –3% to –4%) based on BLS projections and new structural headwinds (remote work, automation), predicting a more pronounced decline.

Final Best-Estimate  
–3% change in employment share (i.e. a 3% relative decline from 2025 to 2030)  
Confidence: Medium

Key Caveats  
• Measurement definition: payroll vs. total civilian employment can shift the share.  
• Baselin

In [8]:
# PRODUCTION TARGET: research.py::get_base_rate_research
# Cell 6: End-to-End Orchestrator

async def get_base_rate_research(
    question_text: str,
    resolution_criteria: str = "",
    fine_print: str = "",
    research_context: str = "",
    model_a: str = "openai/gpt-5.2",
    model_b: str = "anthropic/claude-opus-4-6",
    synthesizer_model: str = "openai/o4-mini",
    use_web_search: bool = False,
    temperature: float = 0.3,
    timeout: int = 60,
) -> str:
    """
    Multi-model base rate researcher. Independently queries two frontier
    models for base rate analysis, then synthesizes with a third model.
    Returns a markdown string to append to research context.

    Args:
        research_context: News/research text (e.g. from AskNews) to include
            in the base rate prompt so models can ground their analysis in
            current evidence.
    """
    # Append :online suffix for web-grounded search
    if use_web_search:
        model_a = model_a + ":online" if ":online" not in model_a else model_a
        model_b = model_b + ":online" if ":online" not in model_b else model_b

    prompt = _build_base_rate_prompt(
        question_text, resolution_criteria, fine_print, research_context
    )

    # Query two frontier models in parallel
    analysis_a, analysis_b = await asyncio.gather(
        _query_model(prompt, model_a, temperature, timeout),
        _query_model(prompt, model_b, temperature, timeout),
    )

    # Synthesize with summarizer
    synthesis = await _synthesize_analyses(
        analysis_a, analysis_b, question_text,
        synthesizer_model, temperature=0.2, timeout=40,
    )

    return f"\n\n## Base Rate Research (Multi-Model)\n{synthesis}"

# Test: full end-to-end on first question (with research context)
if test_questions:
    q = test_questions[0]
    print(f"Running full pipeline for Q{q.id_of_question}: {q.question_text[:60]}...")
    start = time.time()
    result = await get_base_rate_research(
        question_text=q.question_text,
        resolution_criteria=q.resolution_criteria,
        fine_print=q.fine_print,
        research_context=research,
        model_a=MODEL_A,
        model_b=MODEL_B,
        synthesizer_model=SYNTHESIZER,
        use_web_search=USE_WEB_SEARCH,
        temperature=TEMPERATURE,
        timeout=TIMEOUT,
    )
    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s ({len(result)} chars)")
    print(result)


Running full pipeline for Q42562: What will the percent change in US employment share of janit...
Done in 54.9s (1345 chars)


## Base Rate Research (Multi-Model)
Consensus Base-Rate Estimate  
Both analyses agree that the US employment share of janitors and cleaners is likely to decline modestly over 2025–2030, with a 5-year percent change in the roughly –2% to –5% range (relative to the 2025 share).

Key Disagreements  
• Probability of a share increase: Analysis A cites ~30% historical odds; Analysis B estimates ~10–15%.  
• Tail risks: Analysis A’s interquartile range reaches down to –8% (and ≤–10% in rare cases); Analysis B views steeper drops (≤–5%) as less common (15–20% probability).  
• Drivers emphasized: A stresses broad occupational-mix drift and cyclical swings; B highlights accelerating automation and post-pandemic workspace changes.

Final Best-Estimate  
• Point estimate: –3.5% change in janitors/cleaners employment share by 2030 (relative to 2025).  
• Confidence: medi

In [9]:
# EXPLORATION ONLY
# Cell 7: Single-Question Deep Dive — Side-by-Side Raw Analyses
# Run all test questions through the pipeline, capturing intermediate results

deep_dive_results = []

for q in test_questions:
    print(f"\n{'='*80}")
    print(f"Q{q.id_of_question}: {q.question_text[:70]}")
    print(f"{'='*80}")
    
    prompt = _build_base_rate_prompt(q.question_text, q.resolution_criteria, q.fine_print, research)
    
    # Query both models in parallel
    start = time.time()
    a_result, b_result = await asyncio.gather(
        _query_model(prompt, MODEL_A),
        _query_model(prompt, MODEL_B),
    )
    parallel_time = time.time() - start
    
    # Synthesize
    start = time.time()
    synth = await _synthesize_analyses(a_result, b_result, q.question_text, SYNTHESIZER)
    synth_time = time.time() - start
    
    deep_dive_results.append({
        'question_id': q.id_of_question,
        'question_text': q.question_text,
        'model_a_response': a_result,
        'model_b_response': b_result,
        'synthesis': synth,
        'parallel_time_s': parallel_time,
        'synth_time_s': synth_time,
    })
    
    print(f"\n--- {MODEL_A} ({len(a_result)} chars) ---")
    print(a_result[:600])
    print(f"\n--- {MODEL_B} ({len(b_result)} chars) ---")
    print(b_result[:600])
    print(f"\n--- Synthesis by {SYNTHESIZER} ---")
    print(synth)
    print(f"\nTiming: parallel={parallel_time:.1f}s, synthesis={synth_time:.1f}s, total={parallel_time+synth_time:.1f}s")

print(f"\nCompleted deep dive on {len(deep_dive_results)} questions.")


Q42562: What will the percent change in US employment share of janitors and cl

--- openai/gpt-5.2 (7371 chars) ---
## 1) Most relevant reference class(es)

**Primary reference class (best match):**
- **US employment share time series for “Janitors and Cleaners”** (typically BLS SOC **37-2011 Janitors and Cleaners, Except Maids and Housekeeping Cleaners**) measured as  
  \[
  \text{share}_t=\frac{\text{employment in occupation}_t}{\text{total US employment}_t}
  \]
  using either **CPS (household survey occupation shares)** or **OEWS/OES (employer survey occupation employment)**.

**Secondary / robustness reference classes:**
- **Similar large, in-person, low-automation service occupations** with comparabl

--- anthropic/claude-opus-4-6 (4112 chars) ---
# Base Rate Analysis: Percent Change in US Employment Share of Janitors and Cleaners (2025 to 2030)

## Note on Research Context
The retrieved research articles are entirely about NHL hockey and are not relevant to this question. I wi

In [10]:
# EXPLORATION ONLY
# Cell 8: Summarizer A/B Test — o4-mini vs Claude Sonnet 4.6

synth_ab_results = []

for dd in deep_dive_results:
    q_text = dd['question_text']
    print(f"\nQ{dd['question_id']}: {q_text[:60]}...")
    
    # Already have o4-mini synthesis from deep dive
    synth_o4 = dd['synthesis']
    
    # Run Claude Sonnet 4.6 synthesis
    start = time.time()
    synth_sonnet = await _synthesize_analyses(
        dd['model_a_response'], dd['model_b_response'],
        q_text, SYNTHESIZER_ALT,
    )
    sonnet_time = time.time() - start
    
    synth_ab_results.append({
        'question_id': dd['question_id'],
        'o4_mini': synth_o4,
        'sonnet': synth_sonnet,
        'sonnet_time_s': sonnet_time,
    })
    
    print(f"  o4-mini ({len(synth_o4)} chars):")
    print(f"  {synth_o4[:200]}...")
    print(f"  Sonnet 4.6 ({len(synth_sonnet)} chars, {sonnet_time:.1f}s):")
    print(f"  {synth_sonnet[:200]}...")

print("\n--- Manual Scoring ---")
print("Score each synthesis 1-5 on: conciseness, accuracy, usefulness, neutrality")


Q42562: What will the percent change in US employment share of janit...
  o4-mini (1481 chars):
  Consensus base-rate: Both analyses agree that 5-year changes in the employment share of janitors and cleaners are typically modest—usually single‐digit percent moves (most often within ±5 %).

Points ...
  Sonnet 4.6 (1978 chars, 12.6s):
  ## Synthesis

### Areas of Agreement
Both analyses agree on:
- The relevant reference class (BLS SOC 37-2011, employment share over 5-year windows)
- Historical changes tend to be **modest in magnitud...

--- Manual Scoring ---
Score each synthesis 1-5 on: conciseness, accuracy, usefulness, neutrality


In [11]:
# EXPLORATION ONLY
# Cell 9: Quality Assessment Matrix

print("Quality Assessment Matrix")
print("Score each 1-5 after reviewing the full outputs above.\n")

print(f"{'Q ID':<8} {'Ref Class':>10} {'Rate Acc':>10} {'Disagree':>10} {'Synth Q':>10}")
print("-" * 55)
for dd in deep_dive_results:
    print(f"{dd['question_id']:<8} {'__/5':>10} {'__/5':>10} {'__/5':>10} {'__/5':>10}")

print("\nCriteria:")
print("  Ref Class: Did models identify correct, specific reference classes?")
print("  Rate Acc:  Are the stated rates verifiable / plausible?")
print("  Disagree:  When models disagreed, was the disagreement informative?")
print("  Synth Q:   Did the synthesizer accurately compare and summarize?")

Quality Assessment Matrix
Score each 1-5 after reviewing the full outputs above.

Q ID      Ref Class   Rate Acc   Disagree    Synth Q
-------------------------------------------------------
42562          __/5       __/5       __/5       __/5

Criteria:
  Ref Class: Did models identify correct, specific reference classes?
  Rate Acc:  Are the stated rates verifiable / plausible?
  Disagree:  When models disagreed, was the disagreement informative?
  Synth Q:   Did the synthesizer accurately compare and summarize?


In [12]:
# EXPLORATION ONLY
# Cell 10: :online Variant Testing (Web-Grounded Search)
# Queries each model individually to identify which (if any) times out

ONLINE_TIMEOUT = 120  # seconds

online_results = []

for q in test_questions:
    print(f"\nQ{q.id_of_question}: {q.question_text[:60]}...")
    
    prompt = _build_base_rate_prompt(
        q.question_text, q.resolution_criteria, q.fine_print, research
    )
    
    # Append :online suffix
    model_a_online = MODEL_A + ":online" if ":online" not in MODEL_A else MODEL_A
    model_b_online = MODEL_B + ":online" if ":online" not in MODEL_B else MODEL_B
    
    # --- Model A :online ---
    a_result = None
    print(f"  {model_a_online}...", end=" ")
    start_a = time.time()
    try:
        a_result = await _query_model(prompt, model_a_online, TEMPERATURE, ONLINE_TIMEOUT)
        elapsed_a = time.time() - start_a
        print(f"OK ({elapsed_a:.1f}s, {len(a_result)} chars)")
    except TimeoutError:
        elapsed_a = time.time() - start_a
        print(f"TIMEOUT ({elapsed_a:.1f}s)")
    except Exception as e:
        elapsed_a = time.time() - start_a
        print(f"ERROR ({elapsed_a:.1f}s): {e}")
    
    # --- Model B :online ---
    b_result = None
    print(f"  {model_b_online}...", end=" ")
    start_b = time.time()
    try:
        b_result = await _query_model(prompt, model_b_online, TEMPERATURE, ONLINE_TIMEOUT)
        elapsed_b = time.time() - start_b
        print(f"OK ({elapsed_b:.1f}s, {len(b_result)} chars)")
    except TimeoutError:
        elapsed_b = time.time() - start_b
        print(f"TIMEOUT ({elapsed_b:.1f}s)")
    except Exception as e:
        elapsed_b = time.time() - start_b
        print(f"ERROR ({elapsed_b:.1f}s): {e}")
    
    # --- Synthesize (only if both succeeded) ---
    synthesis = None
    if a_result and b_result:
        print(f"  Synthesizing with {SYNTHESIZER}...", end=" ")
        start_s = time.time()
        try:
            synthesis = await _synthesize_analyses(
                a_result, b_result, q.question_text, SYNTHESIZER
            )
            elapsed_s = time.time() - start_s
            print(f"OK ({elapsed_s:.1f}s)")
        except TimeoutError:
            elapsed_s = time.time() - start_s
            print(f"TIMEOUT ({elapsed_s:.1f}s)")
    elif a_result or b_result:
        # Only one model succeeded — use its result directly
        synthesis = a_result or b_result
        print(f"  Skipping synthesis (only one model responded)")
    else:
        print(f"  Skipping synthesis (both models failed)")
    
    online_results.append({
        'question_id': q.id_of_question,
        'model_a_result': a_result,
        'model_b_result': b_result,
        'synthesis': synthesis,
        'model_a_time_s': elapsed_a,
        'model_b_time_s': elapsed_b,
    })

# Compare: training-data-only vs :online
print("\n\n=== COMPARISON: Training-Data vs :online ===")
for dd, ol in zip(deep_dive_results, online_results):
    print(f"\nQ{dd['question_id']}:")
    print(f"  Training-data synthesis ({len(dd['synthesis'])} chars): {dd['synthesis'][:150]}...")
    a_status = f"{len(ol['model_a_result'])} chars in {ol['model_a_time_s']:.1f}s" if ol['model_a_result'] else f"FAILED ({ol['model_a_time_s']:.1f}s)"
    b_status = f"{len(ol['model_b_result'])} chars in {ol['model_b_time_s']:.1f}s" if ol['model_b_result'] else f"FAILED ({ol['model_b_time_s']:.1f}s)"
    print(f"  :online Model A: {a_status}")
    print(f"  :online Model B: {b_status}")
    if ol['synthesis']:
        print(f"  :online synthesis: {ol['synthesis'][:150]}...")
    else:
        print(f"  :online synthesis: N/A")



Q42562: What will the percent change in US employment share of janit...
TIMEOUT (120.0s):online... 
OK (66.0s, 7860 chars)s-4-6:online... 
  Skipping synthesis (only one model responded)


=== COMPARISON: Training-Data vs :online ===

Q42562:
  Training-data synthesis (1481 chars): Consensus base-rate: Both analyses agree that 5-year changes in the employment share of janitors and cleaners are typically modest—usually single‐digi...
  :online Model A: FAILED (120.0s)
  :online Model B: 7860 chars in 66.0s
  :online synthesis: 

# Base Rate Analysis: Percent Change in US Employment Share of Janitors and Cleaners (2025 to 2030)

The provided research context is entirely about...


In [13]:
# EXPLORATION ONLY
# Cell 12: Cost & Latency Tracking
# Estimates OpenRouter API cost per model based on approximate token counts

# Approximate pricing (per 1M tokens, as of March 2026)
# Source: https://openrouter.ai/models
MODEL_PRICING = {
    "openrouter/openai/gpt-5.2":                {"input": 2.00, "output": 8.00},
    "openrouter/openai/gpt-5.2:online":          {"input": 2.00, "output": 8.00},  # :online same base price + search cost
    "openrouter/anthropic/claude-opus-4-6":      {"input": 15.00, "output": 75.00},
    "openrouter/anthropic/claude-opus-4-6:online": {"input": 15.00, "output": 75.00},
    "openrouter/openai/o4-mini":                 {"input": 1.10, "output": 4.40},
    "openrouter/anthropic/claude-sonnet-4-6":    {"input": 3.00, "output": 15.00},
}

# Rough token estimate: 1 token ~ 4 chars
CHARS_PER_TOKEN = 4

def estimate_cost(model, input_chars, output_chars):
    """Estimate cost in USD for a single LLM call."""
    pricing = MODEL_PRICING.get(model, {"input": 5.00, "output": 15.00})
    input_tokens = input_chars / CHARS_PER_TOKEN
    output_tokens = output_chars / CHARS_PER_TOKEN
    return {
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'cost_usd': (input_tokens * pricing['input'] + output_tokens * pricing['output']) / 1_000_000,
        'model': model,
    }

# Aggregate costs from deep dive results
print("=== Cost Breakdown by Model ===")
print(f"{'Model':<50} {'Calls':>6} {'Input Tok':>10} {'Output Tok':>11} {'Cost ($)':>10}")
print("-" * 92)

model_totals = {}
for dd in deep_dive_results:
    prompt = _build_base_rate_prompt(dd['question_text'])
    prompt_chars = len(prompt)
    
    for model_name, response in [(MODEL_A, dd['model_a_response']), (MODEL_B, dd['model_b_response'])]:
        c = estimate_cost(model_name, prompt_chars, len(response))
        if model_name not in model_totals:
            model_totals[model_name] = {'calls': 0, 'input_tokens': 0, 'output_tokens': 0, 'cost_usd': 0}
        model_totals[model_name]['calls'] += 1
        model_totals[model_name]['input_tokens'] += c['input_tokens']
        model_totals[model_name]['output_tokens'] += c['output_tokens']
        model_totals[model_name]['cost_usd'] += c['cost_usd']
    
    # Synthesizer cost (input = both analyses + prompt overhead, output = synthesis)
    synth_input_chars = len(dd['model_a_response']) + len(dd['model_b_response']) + 200  # prompt overhead
    c = estimate_cost(SYNTHESIZER, synth_input_chars, len(dd['synthesis']))
    if SYNTHESIZER not in model_totals:
        model_totals[SYNTHESIZER] = {'calls': 0, 'input_tokens': 0, 'output_tokens': 0, 'cost_usd': 0}
    model_totals[SYNTHESIZER]['calls'] += 1
    model_totals[SYNTHESIZER]['input_tokens'] += c['input_tokens']
    model_totals[SYNTHESIZER]['output_tokens'] += c['output_tokens']
    model_totals[SYNTHESIZER]['cost_usd'] += c['cost_usd']

# Add Sonnet A/B test costs
for ab in synth_ab_results:
    # Find matching deep dive for input chars
    dd_match = next(dd for dd in deep_dive_results if dd['question_id'] == ab['question_id'])
    synth_input_chars = len(dd_match['model_a_response']) + len(dd_match['model_b_response']) + 200
    c = estimate_cost(SYNTHESIZER_ALT, synth_input_chars, len(ab['sonnet']))
    if SYNTHESIZER_ALT not in model_totals:
        model_totals[SYNTHESIZER_ALT] = {'calls': 0, 'input_tokens': 0, 'output_tokens': 0, 'cost_usd': 0}
    model_totals[SYNTHESIZER_ALT]['calls'] += 1
    model_totals[SYNTHESIZER_ALT]['input_tokens'] += c['input_tokens']
    model_totals[SYNTHESIZER_ALT]['output_tokens'] += c['output_tokens']
    model_totals[SYNTHESIZER_ALT]['cost_usd'] += c['cost_usd']

grand_total = 0.0
for model_name, totals in sorted(model_totals.items()):
    print(f"{model_name:<50} {totals['calls']:>6} {totals['input_tokens']:>10.0f} {totals['output_tokens']:>11.0f} ${totals['cost_usd']:>9.4f}")
    grand_total += totals['cost_usd']

print(f"\n{'GRAND TOTAL':>79} ${grand_total:.4f}")
print(f"\nNote: Token counts approximate (chars/4). :online calls may incur additional search fees.")

# Latency summary
print("\n=== Latency Summary ===")
for dd in deep_dive_results:
    total = dd['parallel_time_s'] + dd['synth_time_s']
    print(f"Q{dd['question_id']}: parallel={dd['parallel_time_s']:.1f}s + synth={dd['synth_time_s']:.1f}s = {total:.1f}s")

=== Cost Breakdown by Model ===
Model                                               Calls  Input Tok  Output Tok   Cost ($)
--------------------------------------------------------------------------------------------
anthropic/claude-opus-4-6                               1        164        1028 $   0.0162
anthropic/claude-sonnet-4-6                             1       2921         494 $   0.0220
openai/gpt-5.2                                          1        164        1843 $   0.0285
openai/o4-mini                                          1       2921         370 $   0.0202

                                                                    GRAND TOTAL $0.0869

Note: Token counts approximate (chars/4). :online calls may incur additional search fees.

=== Latency Summary ===
Q42562: parallel=50.0s + synth=9.2s = 59.2s


## Cell 13: Summary & Recommendations

Fill in after running all cells:

**Best summarizer model**: ___ (o4-mini vs Claude Sonnet 4.6)

**Best frontier configuration**: ___ (training-data-only vs `:online`)

**Overall**: Which configuration maximizes quality per dollar?

**Recommended defaults for `research.py`**:
- `model_a` = ___
- `model_b` = timed out at 120 s
- `synthesizer_model` = ___
- `use_web_search` = ___
- `temperature` = ___

### Transfer to Production
Cells 3-6 copy directly to `research.py` with no changes. Update the module-level default
model constants based on the recommendations above. See Part 7b of the planning doc for the
full Code Transfer Checklist.